# Solving Pickle Doomsday
**Converting OGGM pickles to zarr**

In [1]:
from oggm import cfg, utils
from oggm import workflow
from oggm.core import gis
from oggm import DEFAULT_BASE_URL
import oggm.utils.geozarr as geozarr
import xarray as xr
import shapely
from shapely.geometry import mapping
import numpy as np
import rioxarray as rio

In [2]:
cfg.initialize(logging_level="ERROR")

cfg.PARAMS["use_multiprocessing"] = True
cfg.PARAMS["continue_on_error"] = True
cfg.PATHS["working_dir"] = utils.gettempdir(dirname="oggm-geozarr", reset=False)
rgi_ids = ["RGI60-11.00897"]#, "RGI60-06.00377"]  # HEF, Bruarjoekull

DEFAULT_BASE_URL

2026-05-20 14:59:40: oggm.cfg: Reading default parameters from the OGGM `params.cfg` configuration file.
2026-05-20 14:59:40: oggm.cfg: Multiprocessing switched OFF according to the parameter file.
2026-05-20 14:59:40: oggm.cfg: Multiprocessing: using all available processors (N=22)
2026-05-20 14:59:40: oggm.cfg: Multiprocessing switched ON after user settings.
2026-05-20 14:59:40: oggm.cfg: PARAMS['continue_on_error'] changed from `False` to `True`.


'https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L3-L5_files/2025.6/elev_bands/W5E5/per_glacier_spinup/'

Use L4 files as they contain the most pickles.

In [3]:
# base_url="https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L1-L2_files/2025.6/elev_bands/"
gdirs = workflow.init_glacier_directories(
    rgi_ids,
    prepro_base_url=DEFAULT_BASE_URL,  # where to fetch the data?
    from_prepro_level=4,  # what kind of data?
    prepro_border=80,  # how big of a map?
)
gdir = gdirs[0]

2026-05-20 14:59:40: oggm.workflow: init_glacier_directories from prepro level 4 on 1 glaciers.
2026-05-20 14:59:40: oggm.workflow: Execute entity tasks [gdir_from_prepro] on 1 glaciers


Add geometries pickle as this is not included in any file level:

In [4]:
workflow.execute_entity_task(gis.glacier_masks, gdir);

2026-05-20 14:59:40: oggm.workflow: Execute entity tasks [glacier_masks] on 1 glaciers


In [5]:
pickle_paths = geozarr.get_pickle_paths(gdir.dir)
pickle_paths

[PosixPath('geometries.pkl'),
 PosixPath('inversion_flowlines.pkl'),
 PosixPath('inversion_input.pkl'),
 PosixPath('downstream_line.pkl'),
 PosixPath('model_flowlines.pkl'),
 PosixPath('inversion_output.pkl'),
 PosixPath('model_flowlines_dyn_melt_f_calib.pkl')]

In [6]:
pickle_data = geozarr.get_pickle_data(pickle_paths, gdir, type_only=False)
pickle_types = geozarr.get_pickle_data(pickle_paths, gdir, type_only=True)

model_flowlines_dyn_melt_f_calib not in cfg.BASENAMES.
Pickle model_flowlines_dyn_melt_f_calib of type <class 'str'> not parseable.
model_flowlines_dyn_melt_f_calib not in cfg.BASENAMES.
Pickle model_flowlines_dyn_melt_f_calib of type <class 'str'> not parseable.


## Convert between pickles and zarrs

Currently OGGM supports full conversion between pickle and zarr.
For more complex pickles, helper functions coerce certain types into zarr-compatible structures.
If the zarr doesn't exist, `read_store` will fall back to pickles.

In [7]:
inversion_input = gdir.read_store("inversion_input")

/home/gampnico/Documents/WORK/OGGM/oggm-gampnico/oggm/utils/_workflow.py:3407: UserWarning: Zarr not found, attempting to read pickle file instead.
  warnings.warn(


# More complex pickles

Let's now tackle more complex pickles. `downstream_line` contains a shapely LINESTRING object

In [8]:
pickle = gdir.read_pickle("downstream_line")
ds = geozarr.convert_pickles_to_datatree({"downstream_line":pickle, "inversion_input":inversion_input})
ds

<xarray.DataTree>
Group: /
├── Group: /downstream_line
│   │   Dimensions:          (x: 50, y: 2, bedshapes: 50, surface_h: 50, w0s: 50)
│   │   Coordinates:
│   │     * bedshapes        (bedshapes) float64 400B 0.001991 0.002011 ... 0.001758
│   │     * surface_h        (surface_h) float64 400B 2.43e+03 2.427e+03 ... 2.153e+03
│   │     * w0s              (w0s) float64 400B 179.0 165.9 176.1 ... 208.9 208.9 208.9
│   │   Dimensions without coordinates: x, y
│   │   Data variables:
│   │       downstream_line  (x, y) float64 800B 196.0 80.0 197.4 ... 38.33 276.1 36.92
│   └── Group: /downstream_line/full_line
└── Group: /inversion_input
        Dimensions:                (flux_a0: 50, width: 50, slope_angle: 50,
                                    is_rectangular: 50, is_trapezoid: 50, flux: 50,
                                    hgt: 50)
        Coordinates:
          * flux_a0                (flux_a0) float64 400B 1.612e-05 ... 1.877e-05
          * width                  (width) float64 400B 786.9 612.1 ... 695.1 487.5
          * slope_angle            (slope_angle) float64 400B 0.5658 0.651 ... 0.2022
          * is_rectangular         (is_rectangular) bool 50B False False ... False False
          * is_trapezoid           (is_trapezoid) bool 50B True True True ... True True
          * flux                   (flux) float64 400B 0.008457 0.01458 ... 0.0061
          * hgt                    (hgt) float64 400B 3.619e+03 3.563e+03 ... 2.462e+03
        Data variables:
            dx                     float64 8B 100.0
            is_last                bool 1B True
            invert_with_trapezoid  bool 1B True

Internally this calls a helper function that converts a LineString to a DataArray:

In [9]:
data_tree = geozarr.get_downstream_line_from_pkl(pickle)
type(data_tree["downstream_line"])

xarray.core.dataarray.DataArray

Let's write this directly to disk.

In [10]:
gdir.write_zarr(
    data_tree=ds["downstream_line"],
    filename="data_store",
)

Note that if you use `read_zarr`, this outputs the raw zarr file with no validation.
The type for `downstream_line` remains a DataArray.

In [11]:
data_tree = gdir.read_zarr("data_store")
data_tree

<xarray.DataTree>
Group: /
│   Dimensions:          (x: 50, y: 2, bedshapes: 50, surface_h: 50, w0s: 50)
│   Coordinates:
│     * bedshapes        (bedshapes) float64 400B 0.001991 0.002011 ... 0.001758
│     * surface_h        (surface_h) float64 400B 2.43e+03 2.427e+03 ... 2.153e+03
│     * w0s              (w0s) float64 400B 179.0 165.9 176.1 ... 208.9 208.9 208.9
│   Dimensions without coordinates: x, y
│   Data variables:
│       downstream_line  (x, y) float64 800B dask.array<chunksize=(50, 2), meta=np.ndarray>
└── Group: /full_line

But if you use `read_store`, this runs `_validate_store` under the hood which converts `downstream_line` back into the `LineString` type expected by OGGM.

In [12]:
data_tree = gdir.read_store("downstream_line")
data_tree

AttributeError: 'Array' object has no attribute 'groups'

In [ ]:
pickle = gdir.read_pickle("downstream_line")

data_tree = geozarr.get_downstream_line_from_pkl(pickle)
type(data_tree["downstream_line"])

ds = geozarr.convert_pickles_to_datatree({"downstream_line":pickle})
gdir.write_zarr(
    filename="downstream_line",
    data_tree=ds,
)
data_tree = gdir.read_zarr("downstream_line")
data_tree

<xarray.DataTree>
Group: /
│   Dimensions:          (x: 50, y: 2, bedshapes: 50, surface_h: 50, w0s: 50)
│   Coordinates:
│     * bedshapes        (bedshapes) float64 400B 0.001991 0.002011 ... 0.001758
│     * surface_h        (surface_h) float64 400B 2.43e+03 2.427e+03 ... 2.153e+03
│     * w0s              (w0s) float64 400B 179.0 165.9 176.1 ... 208.9 208.9 208.9
│   Dimensions without coordinates: x, y
│   Data variables:
│       downstream_line  (x, y) float64 800B dask.array<chunksize=(50, 2), meta=np.ndarray>
└── Group: /full_line